# Binance BTCUSDT Bar Diagnostics

Load Binance raw trades, normalize them into the project tick schema, construct several bar types, and plot bar/CUSUM/triple-barrier diagnostics.

In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from afml.data import read_tick_csv
from afml.bars import dollar_bars, tick_bars, time_bars, volume_bars
from afml.sampling import cusum_filter
from afml.plotting import plot_bars, plot_cusum_events, plot_triple_barrier_event

In [ ]:
RAW_PATH = PROJECT_ROOT / "data" / "Binance" / "spot" / "daily" / "trades" / "BTCUSDT" / "BTCUSDT-trades-2026-06-03.csv"
NORMALIZED_PATH = PROJECT_ROOT / "data" / "processed" / "binance" / "trades" / "BTCUSDT" / "BTCUSDT-trades-2026-06-03-normalized.csv"

if not NORMALIZED_PATH.exists():
    subprocess.run(
        [sys.executable, str(PROJECT_ROOT / "scripts" / "prepare_binance_trades.py"), str(RAW_PATH)],
        cwd=PROJECT_ROOT,
        check=True,
    )

NORMALIZED_PATH

In [ ]:
NROWS = 200_000

ticks = read_tick_csv(NORMALIZED_PATH, nrows=NROWS)
ticks.head(), ticks.tail(), ticks.shape

In [ ]:
time = time_bars(ticks, "1min")
tick = tick_bars(ticks, threshold=1_000)
volume_threshold = ticks["volume"].sum() / 100
dollar_threshold = ticks["dollar_value"].sum() / 100
volume = volume_bars(ticks, threshold=volume_threshold)
dollar = dollar_bars(ticks, threshold=dollar_threshold)

pd.Series(
    {
        "time_bars": len(time),
        "tick_bars": len(tick),
        "volume_bars": len(volume),
        "dollar_bars": len(dollar),
        "volume_threshold": volume_threshold,
        "dollar_threshold": dollar_threshold,
    }
)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=False)
plot_bars(time, ax=axes[0, 0], title="BTCUSDT time bars")
plot_bars(tick, ax=axes[0, 1], title="BTCUSDT tick bars", x_axis="bar")
plot_bars(volume, ax=axes[1, 0], title="BTCUSDT volume bars", x_axis="bar")
plot_bars(dollar, ax=axes[1, 1], title="BTCUSDT dollar bars", x_axis="bar")
fig.tight_layout()

In [ ]:
close = dollar["close"]
threshold = close.diff().abs().dropna().std() * 3
events = cusum_filter(close, threshold=threshold)

ax = plot_cusum_events(close, events, title="CUSUM events on dollar bars")
ax.figure.set_size_inches(14, 4)
threshold, len(events)

In [ ]:
if len(events) > 0:
    event_time = events[0]
    event_position = close.index.get_loc(event_time)
    vertical_position = min(event_position + 20, len(close.index) - 1)
    target = close.pct_change().rolling(50, min_periods=5).std().bfill().loc[event_time]
    event = pd.Series({"t1": close.index[vertical_position], "trgt": target, "side": 1.0})
    ax = plot_triple_barrier_event(
        close,
        event_time,
        event,
        pt_mult=1,
        sl_mult=1,
        title="Triple-barrier box for first CUSUM event",
    )
    ax.figure.set_size_inches(14, 4)
else:
    print("No CUSUM events found. Try lowering the threshold multiplier.")